In [ ]:
import cv2
import numpy as np
import time
import serial
import time

In [ ]:
PORT = 'COM5' 
BAUD_RATE = 9600
x=600
y=600
origin=(x//2,y//2)
ms1="090"
ms2="000"

In [ ]:
ser = serial.Serial(PORT, BAUD_RATE, timeout=1)
print(f"--- Connected to {PORT} at {BAUD_RATE} baud ---")
print("Type characters to send to STM32 (Ctrl+C to exit)\n")

In [ ]:
def calculator(x,y,r1,r2,origin):
    x=x-origin[0]
    y=y-origin[1]
    phy=2*np.pi-np.arccos((x*x+y*y-r1*r1-r2*r2)/(2*r1*r2))
    a=r1+r2*np.cos(phy)
    b=r2*np.sin(phy)
    theta=np.arcsin(y/np.sqrt(a*a+b*b))-np.atan2(b,a)
    if (x<0):
        theta=np.pi+np.arcsin(-y/np.sqrt(a*a+b*b))-np.atan2(b,a)
    theta=np.tan(np.arctan(theta))
    phy=np.tan(np.arctan(phy))
    if (theta>np.pi/2):
        theta=np.nan
    if (theta<0):
        theta=np.nan
    if (phy>2*np.pi):
        phy=np.nan
    if (phy<np.pi):
        phy=np.nan    
    return (theta,phy) 

In [ ]:
r1=150
r2=150
balancer=np.ones((x,y,3)).astype(np.uint8)*255
for theta in range(0,180):
    cx1=r1*np.cos(np.pi/180*theta)+origin[1]
    cy1=r1*np.sin(np.pi/180*theta)+origin[0]
    for phy in range (180,360):
        cx2=cx1+ r2*np.cos(np.pi/180*(phy+theta))
        cy2=cy1+ r2*np.sin(np.pi/180*(phy+theta))
        th,ph=calculator(cx2,cy2,r1,r2,origin)
        ccx2=r1*np.cos(th)+origin[1]+ r2*np.cos((ph+th))
        ccy2=r1*np.sin(th)+origin[0]+ r2*np.sin((ph+th))
        cv2.circle(balancer,(int(cx2),int(cy2)),2,(0,0,0),2)


In [ ]:
# cv2.imshow("image",balancer)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
image = cv2.imread('image.png')
image=cv2.resize(image,(y,x))
image=cv2.bitwise_or(balancer,image)
h, w = image.shape[:2]
screen = np.ones((h, w, 3), dtype=np.uint8) * 255
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV) # Using INV often helps with white backgrounds
contours, _ = cv2.findContours(thresh, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
for i, cnt in enumerate(contours):
    points = cnt.reshape(-1, 2)
    # time.sleep(0.1)
    # msg="A"+ms1+ms2+"120"
    # ser.write(msg.encode('utf-8'))
    # time.sleep(0.1)
    for point in points:
        try:
            helper = np.ones((h, w, 3), dtype=np.uint8) * 255
            px, py = point 
            theta,phy=calculator(px, py, r1, r2, origin)
            cx1=r1*np.cos(theta)+origin[1]
            cy1=r1*np.sin(theta)+origin[0]
            cv2.line(helper,origin,(int(cx1),int(cy1)),(0,234,10),2)
            # .........................................................
            print(180-theta*180/np.pi,phy*180/np.pi-180)
            ms1=str(int(180-theta*180/np.pi)).zfill(3)
            ms2=str(int(-180+phy*180/np.pi)).zfill(3)
            msg="A"+ms1+ms2+"000"
            ser.write(msg.encode('utf-8'))
            # .........................................................
            cx2=cx1+ r2*np.cos((phy+theta))
            cy2=cy1+ r2*np.sin((phy+theta))
            cv2.line(helper,(int(cx1),int(cy1)),(int(cx2),int(cy2)),(0,234,10),2)
            cv2.circle(screen, (int(cx2), int(cy2)), 1, (0, 255, 0), -1)
            helper=cv2.bitwise_and(helper,screen)
            cv2.imshow("Tracing Boundary", helper)
            cv2.imshow("Tracing", balancer)
            
            if cv2.waitKey(50) & 0xFF == ord('q'):
                break
        except:
            continue
cv2.destroyAllWindows()        

In [ ]:
# image = cv2.imread('image.png')
# image=cv2.resize(image,(y,x))
# fft=image.copy()
# # image=cv2.bitwise_or(balancer,image)
# h, w = image.shape[:2]
# screen = np.ones((h, w, 3), dtype=np.uint8) * 255
# gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
# flag=0
# for i in range (0,x):
#     if (flag==1):
#         break
#     for j in range (0,y):
#         # try:
#             # i=input()
#             # j=input()
#             helper = np.ones((h, w, 3), dtype=np.uint8) * 255
#             theta,phy=calculator(i, j, r1, r2, origin)
#             if np.isnan(theta) or np.isnan(phy):
#                 continue          
#             print(180-theta*180/np.pi,phy*180/np.pi-180)
#             cx1=r1*np.cos(theta)+origin[1]
#             cy1=r1*np.sin(theta)+origin[0]
#             cv2.line(helper,origin,(int(cx1),int(cy1)),(0,234,10),2)
#             cx2=cx1+ r2*np.cos((phy+theta))
#             cy2=cy1+ r2*np.sin((phy+theta))
#             cv2.line(helper,(int(cx1),int(cy1)),(int(cx2),int(cy2)),(0,234,10),2)
#             cv2.circle(screen, (int(cx2), int(cy2)), 1, (0, 255, 0), -1)
#             helper=cv2.bitwise_and(helper,screen)
#             fft=helper
#             cv2.imshow("Tracing Boundary", helper)
#             if cv2.waitKey(1) & 0xFF == ord('q'):
#                 flag=1
#                 break
#         # except:
#         #     continue
# cv2.imwrite("file.png",fft)        
# cv2.destroyAllWindows()        

In [ ]:
screen.shape